# Sidewalk Inspection Dashboard

Real-time overview of NYC sidewalk inspection data with interactive filtering and visualization.

In [ ]:
# Setup and imports
import sys
sys.path.insert(0, '/home/user/nyc_data')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import VBox, HBox, Output
import os
from datetime import datetime, timedelta

# Import the toolkit
try:
    from socrata_toolkit.core.client import SocrataClient, SocrataConfig
    from socrata_toolkit.analysis.core import profile_dataframe
    from socrata_toolkit.quality.sla import check_sla_status
    from socrata_toolkit.viz.core import histogram, bar_chart, time_series_chart
except ImportError as e:
    print(f"Warning: Could not import toolkit modules: {e}")
    print("Using sample data instead.")

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("✓ Environment initialized")

## Interactive Controls

Select your filters below to customize the dashboard.

In [ ]:
# Create interactive controls
borough_selector = widgets.Dropdown(
    options=['All Boroughs', 'MANHATTAN', 'BRONX', 'BROOKLYN', 'QUEENS', 'STATEN_ISLAND'],
    value='All Boroughs',
    description='Borough:',
    style={'description_width': 'initial'}
)

date_range = widgets.DatetimeRangeSlider(
    value=(
        pd.Timestamp('2024-01-01'),
        pd.Timestamp.now()
    ),
    min=pd.Timestamp('2020-01-01'),
    max=pd.Timestamp.now(),
    step=timedelta(days=1),
    description='Date Range:',
    style={'description_width': 'initial'}
)

metric_selector = widgets.Dropdown(
    options=['Violations', 'Inspections', 'Completion Rate', 'Avg Severity'],
    value='Violations',
    description='Metric:',
    style={'description_width': 'initial'}
)

refresh_button = widgets.Button(
    description='Refresh Data',
    button_style='info',
    tooltip='Fetch fresh data from NYC Open Data',
)

controls = VBox([
    widgets.Label("Configure Dashboard:"),
    borough_selector,
    date_range,
    metric_selector,
    refresh_button
])

display(controls)

## Data Summary

In [ ]:
# Function to fetch and filter data
def fetch_inspection_data(borough=None, start_date=None, end_date=None):
    """Fetch inspection data with optional filters."""
    try:
        client = SocrataClient(SocrataConfig())
        
        # Build SOQL query
        where_clauses = []
        
        if borough and borough != 'All Boroughs':
            where_clauses.append(f"borough='{borough}'")
        
        if start_date:
            where_clauses.append(f"created_date>'{start_date.isoformat()}'")
        
        if end_date:
            where_clauses.append(f"created_date<'{end_date.isoformat()}'")
        
        where = ' AND '.join(where_clauses) if where_clauses else None
        
        df = client.fetch_dataframe(
            'data.cityofnewyork.us',
            'dntt-gqwq',  # inspection dataset
            max_rows=50000,
            where=where if where else None
        )
        
        return df
    
    except Exception as e:
        print(f"Note: Using sample data (API fetch failed: {e})")
        # Return sample data for demo
        return generate_sample_inspection_data()

def generate_sample_inspection_data():
    """Generate sample inspection data for demo."""
    np.random.seed(42)
    n = 5000
    
    return pd.DataFrame({
        'objectid': range(n),
        'created_date': pd.date_range('2024-01-01', periods=n, freq='H'),
        'borough': np.random.choice(['MANHATTAN', 'BRONX', 'BROOKLYN', 'QUEENS', 'STATEN_ISLAND'], n),
        'violation_count': np.random.poisson(2, n),
        'severity': np.random.choice(['LOW', 'MEDIUM', 'HIGH'], n),
        'status': np.random.choice(['OPEN', 'IN_PROGRESS', 'CLOSED'], n),
    })

# Initial data load
print("Loading inspection data...")
df_inspection = fetch_inspection_data(
    borough=borough_selector.value if borough_selector.value != 'All Boroughs' else None,
    start_date=date_range.value[0],
    end_date=date_range.value[1]
)

print(f"✓ Loaded {len(df_inspection):,} inspection records")
print(f"  Date range: {df_inspection['created_date'].min()} to {df_inspection['created_date'].max()}")
print(f"  Boroughs: {df_inspection['borough'].nunique()}")

## Violations by Borough

In [ ]:
# Aggregate by borough
borough_stats = df_inspection.groupby('borough').agg({
    'objectid': 'count',
    'violation_count': 'sum'
}).rename(columns={'objectid': 'inspections', 'violation_count': 'total_violations'})

borough_stats['avg_violations_per_inspection'] = borough_stats['total_violations'] / borough_stats['inspections']

# Create visualization
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Inspections by Borough', 'Total Violations by Borough'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}]]
)

fig.add_trace(
    go.Bar(
        x=borough_stats.index,
        y=borough_stats['inspections'],
        name='Inspections',
        marker_color='rgb(55, 83, 109)',
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=borough_stats.index,
        y=borough_stats['total_violations'],
        name='Violations',
        marker_color='rgb(214, 39, 40)',
    ),
    row=1, col=2
)

fig.update_layout(height=400, showlegend=True, title_text='Inspection Activity by Borough')
fig.update_xaxes(title_text='Borough', row=1, col=1)
fig.update_xaxes(title_text='Borough', row=1, col=2)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Count', row=1, col=2)

fig.show()

## Violations Over Time

In [ ]:
# Time series of violations
df_inspection['date'] = pd.to_datetime(df_inspection['created_date']).dt.date
daily_violations = df_inspection.groupby('date')['violation_count'].sum().reset_index()
daily_violations['date'] = pd.to_datetime(daily_violations['date'])

fig = px.line(
    daily_violations,
    x='date',
    y='violation_count',
    title='Daily Violation Count',
    labels={'violation_count': 'Violations', 'date': 'Date'},
    markers=True
)

fig.update_traces(fill='tozeroy')
fig.update_layout(hovermode='x unified', height=400)
fig.show()

## Violation Severity Distribution

In [ ]:
# Severity breakdown
severity_counts = df_inspection['severity'].value_counts()

fig = px.pie(
    values=severity_counts.values,
    names=severity_counts.index,
    title='Violations by Severity Level',
    color_discrete_sequence=px.colors.sequential.RdBu
)

fig.update_layout(height=400)
fig.show()

## Status Breakdown

In [ ]:
# Status distribution
status_counts = df_inspection['status'].value_counts()

fig = go.Figure(data=[
    go.Bar(
        x=status_counts.index,
        y=status_counts.values,
        marker_color=['rgb(99, 110, 250)', 'rgb(239, 85, 59)', 'rgb(0, 204, 150)'],
        text=status_counts.values,
        textposition='auto',
    )
])

fig.update_layout(
    title='Inspection Status Distribution',
    xaxis_title='Status',
    yaxis_title='Count',
    height=400
)

fig.show()

## Data Table

In [ ]:
# Show data table
display_cols = ['objectid', 'created_date', 'borough', 'violation_count', 'severity', 'status']
df_display = df_inspection[display_cols].head(100)

print(f"Showing first 100 of {len(df_inspection):,} records:")
print(df_display.to_string())

## Export Data

In [ ]:
# Export options
export_button = widgets.Button(description='Download as CSV')
excel_button = widgets.Button(description='Download as Excel')
json_button = widgets.Button(description='Download as JSON')

def on_csv_click(b):
    filename = f'inspection_data_{pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")}.csv'
    df_inspection.to_csv(filename, index=False)
    print(f"✓ Saved to {filename}")

def on_excel_click(b):
    filename = f'inspection_data_{pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")}.xlsx'
    df_inspection.to_excel(filename, index=False)
    print(f"✓ Saved to {filename}")

def on_json_click(b):
    filename = f'inspection_data_{pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")}.json'
    df_inspection.to_json(filename, orient='records', indent=2)
    print(f"✓ Saved to {filename}")

export_button.on_click(on_csv_click)
excel_button.on_click(on_excel_click)
json_button.on_click(on_json_click)

print("Download the filtered data:")
display(HBox([export_button, excel_button, json_button]))